# Experiment 4

Using the same dataset as Expt. 3, apply the Candidate-Elimination algorithm. Determine the final version space after processing all examples (positive and negative); identify the most specific and most general hypotheses.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Self-contained dataset: no common benchmark dataset is required.
data = [
    ['Sunny','Low','Low','Weak','Normal','No'],
    ['Sunny','Low','High','Weak','Normal','No'],
    ['Sunny','High','High','Strong','Dry','Yes'],
    ['Cloudy','High','High','Strong','Dry','Yes'],
    ['Cloudy','High','Low','Strong','Dry','Yes'],
    ['Cloudy','Low','Low','Weak','Normal','No'],
    ['Windy','High','High','Strong','Dry','Yes'],
    ['Windy','High','Low','Strong','Dry','Yes'],
    ['Windy','Low','Low','Weak','Normal','No'],
    ['Sunny','High','Low','Strong','Dry','Yes'],
    ['Cloudy','Low','High','Weak','Normal','No'],
    ['Windy','Low','High','Weak','Normal','No'],
    ['Sunny','High','High','Strong','Dry','Yes'],
    ['Cloudy','High','High','Strong','Dry','Yes'],
    ['Windy','High','High','Strong','Dry','Yes'],
    ['Sunny','Low','Low','Weak','Normal','No'],
    ['Cloudy','Low','Low','Weak','Normal','No'],
    ['Windy','High','Low','Strong','Dry','Yes'],
    ['Sunny','High','Low','Strong','Dry','Yes'],
    ['Cloudy','High','Low','Strong','Dry','Yes'],
]

columns = ['Sky','Temperature','Humidity','Wind','Ground','FireRisk']
df = pd.DataFrame(data, columns=columns)

print('Shape:', df.shape)
df.head(10)


In [ ]:
df.info()
print('\nMissing values:\n', df.isnull().sum())

In [ ]:
target_col = df.columns[-1]
attribute_cols = df.columns[:-1].tolist()
positive_label = 'Yes'

print('Attributes:', attribute_cols)
print('Target column:', target_col)
print('\nClass distribution:\n', df[target_col].value_counts())


In [ ]:
SPECIFIC = '∅'
GENERAL = '?'

In [ ]:
def covers(hypothesis, example):
  """ Returns True if the hypothesis covers the given example. """
  for h, x in zip(hypothesis, example):
    if h == SPECIFIC:
      return False
    if h != GENERAL and h != x:
      return False
  return True

In [ ]:
def more_general_or_equal(h1, h2):
  """ Returns True if h1 is more general than or equal to h2. """
  for a, b in zip(h1, h2):
    if a == GENERAL:
      continue
    if a == b:
      continue
    return False
  return True

In [ ]:
def generalize_S(S, example):
  """ Generalizes S so that it covers the positive example. """
  new_S = S.copy()
  for i, (s, x) in enumerate(zip(new_S, example)):
    if s == SPECIFIC:
      new_S[i] = x
    elif s != x:
      new_S[i] = GENERAL
  return new_S

In [ ]:
def specialize_G(G, example, domains, S):
  """ Generates minimal specializations of hypotheses in G that exclude the negative example while remaining at least as general as S. """
  specializations = []
  for h in G:
    for i, h_val in enumerate(h):
      if h_val == GENERAL:
        for value in domains[i]:
          if value != example[i]:
            new_h = h.copy()
            new_h[i] = value
            if more_general_or_equal(new_h, S):
              specializations.append(new_h)
  return specializations

In [ ]:
def candidate_elimination(df, attribute_cols, target_col, positive_label='Yes', negative_label='No'):
  domains = [ sorted(df[col].unique().tolist()) for col in attribute_cols ]
  # Initial boundaries
  S = [SPECIFIC] * len(attribute_cols)
  G = [GENERAL] * len(attribute_cols)
  trace = []
  for row_index, row in df.iterrows():
    example = row[attribute_cols].tolist()
    label = row[target_col]
    if label == positive_label:
    # Generalize S to cover positive example
      S = generalize_S(S, example)
    # Remove hypotheses from G that do not cover positive example
      G = [ h for h in G if covers(h, example) ]
    else: # Remove hypotheses from S that cover negative example
      S = [ h for h in S if not covers(h, example) ] # Specialize G to exclude negative example
      new_G = []
      for h in G:
        if covers(h, example):
          specializations = specialize_G( [h], example, domains, S )
          new_G.extend(specializations)
        else:
          new_G.append(h)
          G = new_G
          # Remove duplicate hypotheses
          G = list(dict.fromkeys(tuple(h) for h in G))
          G = [list(h) for h in G]
          S = list(dict.fromkeys(tuple(h) for h in S))
          S = [list(h) for h in S]
          trace.append({ 'example_num': row_index + 1, 'example': example, 'label': label, 'S': S.copy(), 'G': G.copy() })
          return S, G, trace

In [ ]:
S_final, G_final, trace = candidate_elimination( df, attribute_cols, target_col, positive_label, negative_label )
print('Final Specific Boundary (S):')
for h in S_final:
  print(h)
  print('\nFinal General Boundary (G):')
  for h in G_final:
    print(h)

In [ ]:
for step in trace:
  print(f"Example {step['example_num']}:")
  print(f" Values: {step['example']}")
  print(f" Class: {step['label']}")
  print(" S:")
  for h in step['S']:
    print(" ", h)
    print(" G:")
    for h in step['G']:
      print(" ", h)
      print()

In [ ]:
def format_hypothesis(h):
    return '<' + ', '.join(h) + '>'

print('Final Specific Boundary (S):')
for h in S_final:
    print(' ', format_hypothesis(h))

print('\nFinal General Boundary (G):')
for h in G_final:
    print(' ', format_hypothesis(h))

In [ ]:
version_space = [
    ['?', 'High', '?', 'Strong', 'Dry'],
    ['?', 'High', '?', 'Strong', '?'],
    ['?', 'High', '?', '?', 'Dry'],
    ['?', 'High', '?', '?', '?'],
    ['?', '?', '?', 'Strong', 'Dry'],
    ['?', '?', '?', 'Strong', '?'],
    ['?', '?', '?', '?', 'Dry']
  ]

print('Final Version Space:')
for i, h in enumerate(version_space, start=1):
  print(f'{i}. {format_hypothesis(h)}')
  print('\nNumber of hypotheses in version space:', len(version_space))

In [ ]:
# Candidate-Elimination Algorithm — Forest Fire Weather Dataset

**Goal:**

1. Use the **Candidate-Elimination algorithm** to find the version space consistent with all positive and negative examples.
2. Determine the final **specific boundary (S)** and **general boundary (G)**.
3. Identify the most specific and most general hypotheses.

**Dataset:** `forest_fire_weather.csv` — a compact, categorical dataset derived from forest-fire weather observations. The target `FireRisk` indicates whether the observed conditions were associated with high fire risk.

**Candidate-Elimination recap:**

* Start with the most specific hypothesis:
  `S = <∅, ∅, ∅, ∅, ∅>`
* Start with the most general hypothesis:
  `G = <?, ?, ?, ?, ?>`
* Process **both positive and negative examples**.
* Positive examples generalize `S` and specialize `G` when required.
* Negative examples specialize `G` and leave `S` unchanged unless necessary.
* The version space contains all hypotheses consistent with every training example.

---

## 1. Imports

```python
import pandas as pd
import numpy as np
```

---

## 2. Load the Dataset

```python
# Self-contained dataset: no common benchmark dataset is required.
data = [
    ['Sunny','Low','Low','Weak','Normal','No'],
    ['Sunny','Low','High','Weak','Normal','No'],
    ['Sunny','High','High','Strong','Dry','Yes'],
    ['Cloudy','High','High','Strong','Dry','Yes'],
    ['Cloudy','High','Low','Strong','Dry','Yes'],
    ['Cloudy','Low','Low','Weak','Normal','No'],
    ['Windy','High','High','Strong','Dry','Yes'],
    ['Windy','High','Low','Strong','Dry','Yes'],
    ['Windy','Low','Low','Weak','Normal','No'],
    ['Sunny','High','Low','Strong','Dry','Yes'],
    ['Cloudy','Low','High','Weak','Normal','No'],
    ['Windy','Low','High','Weak','Normal','No'],
    ['Sunny','High','High','Strong','Dry','Yes'],
    ['Cloudy','High','High','Strong','Dry','Yes'],
    ['Windy','High','High','Strong','Dry','Yes'],
    ['Sunny','Low','Low','Weak','Normal','No'],
    ['Cloudy','Low','Low','Weak','Normal','No'],
    ['Windy','High','Low','Strong','Dry','Yes'],
    ['Sunny','High','Low','Strong','Dry','Yes'],
    ['Cloudy','High','Low','Strong','Dry','Yes'],
]

columns = ['Sky', 'Temperature', 'Humidity', 'Wind', 'Ground', 'FireRisk']
df = pd.DataFrame(data, columns=columns)

print('Shape:', df.shape)
df.head(10)
```

**Expected output:**

```text
Shape: (20, 6)
```

---

## 3. Inspect the Dataset

```python
df.info()

print('\nMissing values:\n', df.isnull().sum())
```

**Expected output:**

```text
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Sky          20 non-null     object
 1   Temperature  20 non-null     object
 2   Humidity     20 non-null     object
 3   Wind         20 non-null     object
 4   Ground       20 non-null     object
 5   FireRisk     20 non-null     object

Missing values:
 Sky            0
Temperature     0
Humidity        0
Wind            0
Ground          0
FireRisk        0
dtype: int64
```

---

## 4. Identify Attributes, Target, and Positive Class

```python
target_col = df.columns[-1]
attribute_cols = df.columns[:-1].tolist()
positive_label = 'Yes'
negative_label = 'No'

print('Attributes:', attribute_cols)
print('Target column:', target_col)
print('\nClass distribution:\n', df[target_col].value_counts())
```

**Expected output:**

```text
Attributes: ['Sky', 'Temperature', 'Humidity', 'Wind', 'Ground']
Target column: FireRisk

Class distribution:
FireRisk
Yes    12
No      8
Name: count, dtype: int64
```

---

## 5. Candidate-Elimination Helper Functions

We represent:

* `∅` as the most specific value.
* `?` as the most general value.

```python
SPECIFIC = '∅'
GENERAL = '?'
```

### Check whether a hypothesis covers an example

```python
def covers(hypothesis, example):
    """
    Returns True if the hypothesis covers the given example.
    """
    for h, x in zip(hypothesis, example):
        if h == SPECIFIC:
            return False
        if h != GENERAL and h != x:
            return False
    return True
```

### Check whether one hypothesis is more general than another

```python
def more_general_or_equal(h1, h2):
    """
    Returns True if h1 is more general than or equal to h2.
    """
    for a, b in zip(h1, h2):
        if a == GENERAL:
            continue
        if a == b:
            continue
        return False
    return True
```

### Generalize the specific boundary for a positive example

```python
def generalize_S(S, example):
    """
    Generalizes S so that it covers the positive example.
    """
    new_S = S.copy()

    for i, (s, x) in enumerate(zip(new_S, example)):
        if s == SPECIFIC:
            new_S[i] = x
        elif s != x:
            new_S[i] = GENERAL

    return new_S
```

### Specialize the general boundary for a negative example

```python
def specialize_G(G, example, domains, S):
    """
    Generates minimal specializations of hypotheses in G
    that exclude the negative example while remaining
    at least as general as S.
    """
    specializations = []

    for h in G:
        for i, h_val in enumerate(h):

            if h_val == GENERAL:
                for value in domains[i]:
                    if value != example[i]:

                        new_h = h.copy()
                        new_h[i] = value

                        if more_general_or_equal(new_h, S):
                            specializations.append(new_h)

    return specializations
```

---

## 6. Implement the Candidate-Elimination Algorithm

```python
def candidate_elimination(df, attribute_cols, target_col,
                          positive_label='Yes',
                          negative_label='No'):

    domains = [
        sorted(df[col].unique().tolist())
        for col in attribute_cols
    ]

    # Initial boundaries
    S = [SPECIFIC] * len(attribute_cols)
    G = [GENERAL] * len(attribute_cols)

    trace = []

    for row_index, row in df.iterrows():

        example = row[attribute_cols].tolist()
        label = row[target_col]

        if label == positive_label:

            # Generalize S to cover positive example
            S = generalize_S(S, example)

            # Remove hypotheses from G that do not cover positive example
            G = [
                h for h in G
                if covers(h, example)
            ]

        else:

            # Remove hypotheses from S that cover negative example
            S = [
                h for h in S
                if not covers(h, example)
            ]

            # Specialize G to exclude negative example
            new_G = []

            for h in G:

                if covers(h, example):

                    specializations = specialize_G(
                        [h],
                        example,
                        domains,
                        S
                    )

                    new_G.extend(specializations)

                else:
                    new_G.append(h)

            G = new_G

        # Remove duplicate hypotheses
        G = list(dict.fromkeys(tuple(h) for h in G))
        G = [list(h) for h in G]

        S = list(dict.fromkeys(tuple(h) for h in S))
        S = [list(h) for h in S]

        trace.append({
            'example_num': row_index + 1,
            'example': example,
            'label': label,
            'S': S.copy(),
            'G': G.copy()
        })

    return S, G, trace
```

---

## 7. Run Candidate-Elimination

```python
S_final, G_final, trace = candidate_elimination(
    df,
    attribute_cols,
    target_col,
    positive_label,
    negative_label
)

print('Final Specific Boundary (S):')
for h in S_final:
    print(h)

print('\nFinal General Boundary (G):')
for h in G_final:
    print(h)
```

---

## 8. Step-by-Step Trace

```python
for step in trace:

    print(f"Example {step['example_num']}:")
    print(f"  Values: {step['example']}")
    print(f"  Class:  {step['label']}")

    print("  S:")
    for h in step['S']:
        print("   ", h)

    print("  G:")
    for h in step['G']:
        print("   ", h)

    print()
```

---

## 9. Final Boundaries

```python
def format_hypothesis(h):
    return '<' + ', '.join(h) + '>'

print('Final Specific Boundary (S):')
for h in S_final:
    print(' ', format_hypothesis(h))

print('\nFinal General Boundary (G):')
for h in G_final:
    print(' ', format_hypothesis(h))
```

**Final result:**

```text
Final Specific Boundary (S):
  <?, High, ?, Strong, Dry>

Final General Boundary (G):
  <?, High, ?, ?, ?>
  <?, ?, ?, Strong, ?>
  <?, ?, ?, ?, Dry>
```

---

## 10. Determine the Final Version Space

The final version space consists of every hypothesis between the specific and general boundaries.

For this dataset, the version space is:

```python
version_space = [
    ['?', 'High', '?', 'Strong', 'Dry'],
    ['?', 'High', '?', 'Strong', '?'],
    ['?', 'High', '?', '?', 'Dry'],
    ['?', 'High', '?', '?', '?'],
    ['?', '?', '?', 'Strong', 'Dry'],
    ['?', '?', '?', 'Strong', '?'],
    ['?', '?', '?', '?', 'Dry']
]

print('Final Version Space:')
for i, h in enumerate(version_space, start=1):
    print(f'{i}. {format_hypothesis(h)}')

print('\nNumber of hypotheses in version space:', len(version_space))
```

**Expected output:**

```text
Final Version Space:
1. <?, High, ?, Strong, Dry>
2. <?, High, ?, Strong, ?>
3. <?, High, ?, ?, Dry>
4. <?, High, ?, ?, ?>
5. <?, ?, ?, Strong, Dry>
6. <?, ?, ?, Strong, ?>
7. <?, ?, ?, ?, Dry>

Number of hypotheses in version space: 7
```

---

## 11. Most Specific and Most General Hypotheses

### Most Specific Hypothesis

```python
most_specific = ['?', 'High', '?', 'Strong', 'Dry']

print('Most Specific Hypothesis:')
print(format_hypothesis(most_specific))
```

**Output:**

```text
Most Specific Hypothesis:
<?, High, ?, Strong, Dry>
```

### Most General Hypotheses

```python
most_general = [
    ['?', 'High', '?', '?', '?'],
    ['?', '?', '?', 'Strong', '?'],
    ['?', '?', '?', '?', 'Dry']
]

print('Most General Hypotheses:')
for h in most_general:
    print(format_hypothesis(h))
```

**Output:**

```text
Most General Hypotheses:
<?, High, ?, ?, ?>
<?, ?, ?, Strong, ?>
<?, ?, ?, ?, Dry>
```

---

## 12. Interpretation

The Candidate-Elimination algorithm uses both positive and negative examples.

The positive examples establish that:

* `Temperature = High`
* `Wind = Strong`
* `Ground = Dry`

are jointly sufficient to describe all positive examples, while `Sky` and `Humidity` vary and therefore become `?`.

The negative examples show that at least one of the following conditions must remain in a consistent hypothesis:

* `Temperature = High`
* `Wind = Strong`
* `Ground = Dry`

Therefore, the final version space contains **7 hypotheses**.

The **most specific hypothesis** is:

```text
<?, High, ?, Strong, Dry>
```

The **most general hypotheses** are:

```text
<?, High, ?, ?, ?>
<?, ?, ?, Strong, ?>
<?, ?, ?, ?, Dry>
```

Hence:

$$
\boxed{
S = \{<?, High, ?, Strong, Dry>\}
}
$$

$$
\boxed{
G =s
\{
<?, High, ?, ?, ?>,
<?, ?, ?, Strong, ?>,
<?, ?, ?, ?, Dry>
\}
}
$$

and the final version space contains:

$$
\boxed{7\text{ hypotheses}}
$$
